# Topic modeling

Topic analysis is an NLP technique that analyses a corpus of text by automatically highlighting the most frequently occurring topics. It is an unsupervised learning technique because the number and description of topics are not known in advance. Like other unsupervised learning techniques, topic analysis requires human analysis to assess the quality of the extracted topics and interpret them. Topic analysis is therefore the first step in a more precise and targeted analysis of a corpus of documents. In this practical exercise, we will test several topic analysis techniques on a corpus of articles from the newspaper Le Monde.

In order to perform a more accurate analysis, select a type of article from the database among :
* 'ART': 'arts'
* 'ENT': 'entreprises'
* 'FRA': 'France'
* 'SOC': 'société'
* 'INT': 'International'
* 'SPO': 'sports'



In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# load the corpus
df = pd.read_csv('data/LeMonde2003_9classes.csv.gz')
# select a type of articles to focus the analysis
df = df[df['category'] == 'TODO'].sample(n=1000).reset_index(drop=True)
print (f"Number of articles in the corpus : {df.shape}")
df.head()

## Latent Dirichlet Allocation

[Latent Dirichlet Allocation (LDA)](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.LatentDirichletAllocation.html) is a probabilistic model in which each document is represented as a mixture of topics, and each topic is represented as a distribution over words. For each word in a document, a topic is sampled from the document’s topic distribution, and the word is then sampled from the corresponding topic’s word distribution. The model assumes a bag-of-words representation, meaning that it does not take word order into account and relies only on word frequencies, which can be obtained using vectorization methods such as CountVectorizer. 

The training phase consists of estimating the latent topic distributions and the document–topic proportions from the observed words, typically using approximate Bayesian inference methods such as variational inference or Gibbs sampling.

**Question**:

> * Encode the text of the selected articles using [CountVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html).
> * train a LDA model on the encoded text.
> * Print the 10 most representative words for each topic.
> * In a new cell, add the STOPWORDS list as a parameter of CountVectorizer, retrain the model and print the 10 most representative words for each topic. Observe the changes.
> * In a new cell, lemmatize using the text spacy `fr_core_news_sm` model and train a LDA model on the lemmatized text. Print the 10 most representative words for each topic.  Observe the changes.


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import  LatentDirichletAllocation 
import matplotlib.pyplot as plt
import spacy

#! python -m spacy download fr_core_news_sm

def plot_top_words(model, vectorizer, n_top_words, title, nb_lines=2):
    feature_names = vectorizer.get_feature_names_out()
    fig, axes = plt.subplots(nb_lines, 5, figsize=(30, 30), sharex=True)
    axes = axes.flatten()
    for topic_idx, topic in enumerate(model.components_):
        top_features_ind = topic.argsort()[-n_top_words:]
        top_features = feature_names[top_features_ind]
        weights = topic[top_features_ind]

        ax = axes[topic_idx]
        ax.barh(top_features, weights, height=0.7)
        ax.set_title(f"Topic {topic_idx + 1}", fontdict={"fontsize": 30})
        ax.tick_params(axis="both", which="major", labelsize=20)
        for i in "top right left".split():
            ax.spines[i].set_visible(False)
        fig.suptitle(title, fontsize=40)

    plt.subplots_adjust(top=0.90, bottom=0.05, wspace=0.90, hspace=0.3)
    plt.show()

n_features = 1000
n_topics = 10
STOPWORDS = [x.strip() for x in open('data/stop_word_fr.txt').readlines()]
nlp = spacy.load("fr_core_news_sm", disable=["parser", "ner"])
df['lemmatized_text'] = [" ".join([token.lemma_ for token in doc]) for doc in nlp.pipe(df['text'])]

# Encode the text with CountVectorizer
# YOUR CODE HERE

# Fit LDA model

# visualize the words in topics
plot_top_words(lda, tf_vectorizer, 10, "LDA Topics")

### PyLDAvis
Once an LDA model has been trained, [pyLDAvis](https://github.com/bmabey/pyLDAvis) can be used to visualise and explore the discovered topics interactively. This tool provides a two-dimensional representation of the topics, in which the distance between circles reflects their similarity and the size of each circle corresponds to how prevalent the topic is in the corpus. It also displays the most relevant words for each topic and allows users to adjust a relevance parameter to balance frequency and distinctiveness. This interactive visualisation enables more effective interpretation, evaluation and comparison of topics than simply inspecting lists of keywords.

In [ ]:
import pyLDAvis
import pyLDAvis.lda_model
pyLDAvis.enable_notebook()

pyLDAvis.lda_model.prepare(lda, tf, tf_vectorizer)

## Non-negative matrix factorisation

[Non-negative matrix factorisation (NMF)](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.NMF.html) is a linear algebra-based topic modelling method that decomposes a document-term matrix into two lower-dimensional non-negative matrices, `H` and `W`. The matrix `W` represents the importance of each topic in each document, while the matrix `H` (components_) represents the importance of each word in each topic. Unlike LDA, NMF is an optimisation-based approach that factorises the input matrix by minimising a reconstruction error under non-negativity constraints, rather than a probabilistic generative model. When applied to text data, which is typically represented using [TF-IDF](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html) features, NMF produces interpretable topics characterised by sets of highly weighted words. 

**Question**:

> * Encode the text of the selected articles using `TfidfVectorizer`.
> * [train a NMF model](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.NMF.html#sklearn.decomposition.NMF.fit_transform) on the encoded text.
> * Print the 20 most representative words for each topic.
> * In a new cell, add the STOPWORDS list as a parameter of CountVectorizer, retrain the model and print the 20 most representative words for each topic. Observe the changes.
> * In a new cell, lemmatize using the text spacy `fr_core_news_sm` model and train a LDA model on the lemmatized text. Print the 20 most representative words for each topic.  Observe the changes.
> * Analyse each topic and describe it with label. Create of dictionnary to map each topic to its label

In [ ]:
from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# Encode the text with TF-IDF vecotrize
# YOUR CODE HERE


# Define the NMF model
# YOUR CODE HERE


# Train the NMF
# YOUR CODE HERE


# print the words for each topic
feature_names = tfidf_vectorizer.get_feature_names_out()
n_top_words = 20

for topic_idx, topic in enumerate(nmf.components_):
    top_indices = np.argsort(-topic)[:n_top_words]
    top_words = [feature_names[i] for i in top_indices]
    
    print(f"Topic {topic_idx}:")
    print("  " + ", ".join(top_words))
    print()

topic_label = {}
topic_label[0] = 'Théâtre'
# YOUR CODE HERE



### Document/Topic analysis
For each document, the W matrix gives us the weight of each topic in that document. It is therefore possible to normalise these weights and graphically represent the relative importance of each topic.

**Question**:

> * Normalise the W matrix by dividing it by its sum per row ([np.sum(axis=1)](https://numpy.org/doc/2.3/reference/generated/numpy.sum.html))
> * Extract the index of the column with the highest value ([argmax](https://numpy.org/devdocs/reference/generated/numpy.argmax.html)) and the value (max) and add this data to the dataframe


The following cell displays examples of article text with the dominant theme, its label and its score.

In [ ]:
import numpy as np

# Normalize the W matrix
# YOUR CODE HERE
df["dominant_topic"] = # YOUR CODE HERE
df["topic_score"] = # YOUR CODE HERE

The following cell displays examples of article text with the dominant theme, its label and its score.

**Question**:

> * Analyse the coherence between the text, the dominant topic and the topic distribution.

In [ ]:

import matplotlib.pyplot as plt
import textwrap

topic_names = [f"{k}-{topic_label[k]}" for k in topic_label.keys()]
    
for row in df.sample(n=5, random_state=2026).itertuples():
    
    doc_id = row.Index
    topic_distribution = W_normalized[doc_id]
    dominant_topic = row.dominant_topic
    dominant_score = row.topic_score
    dominant_label = topic_label[dominant_topic]
    
    print(f"\nDocument {doc_id}")
    print(f"Dominant topic: {dominant_topic} - {dominant_label} ({dominant_score:.3f})\n")
    text = textwrap.fill(row.text, width=100)
    print(text[:1200])
    print("\n\n")
    
    # Topic distribution
    plt.figure(figsize=(12,4))
    bars = plt.bar(topic_names, topic_distribution)
    bars[dominant_topic].set_color("darkred")
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Proportion")
    plt.title(f"Topic mixture for document {doc_id}")
    plt.tight_layout()
    plt.show()

## Topic modeling with embeddings : Bertopic

Unlike LDA and NMF, which rely on word frequency, [BERTopic](https://maartengr.github.io/BERTopic/index.html) is a topic modelling approach based on contextual embeddings to identify semantic similarities between documents. BERTopic combines transformer-based sentence embeddings, dimensionality reduction and density-based clustering to discover topics that reflect the semantic structures beyond simple word co-occurrence patterns. In this section, we will explore how BERTopic works and compare its results with those of previously implemented models.

**Question**:

> * Use [SentenceTransformer](https://sbert.net/#sentencetransformers-documentation) to encode the text of the articles using a model adapted to French from [HuggingFace](https://huggingface.co/models?library=sentence-transformers)
> * Create a CountVectorizer using the French stopwords to identify the topic labels
> * Create a BERTopic object with the CountVectorizer as parameter and fit it on the documents
> * Explore the topics with `topic_model.get_topic_info()` and compare them with the topics discovered by NMF.

In [ ]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer

docs = df["text"].tolist()

# Encode the documents with SentenceTransformer
# YOUR CODE HERE

# Create a CountVectorizer with French stop words
# YOUR CODE HERE

# Create the BERTopic model using the vectorizer and fit it
# YOUR CODE HERE

In [ ]:
topic_model.get_topic_info()

In [ ]:
topic_model.visualize_documents(
    docs=docs,
    embeddings=embeddings,
    hide_annotations=True,
    topics=[0, 1, 2, 3, 4, 5,6, 7, 8, 9],
    height=600,
    width=1000
)

## Extra : Topic number analysis

In topic modelling, selecting an appropriate number of topics is crucial because it directly affects the interpretability, granularity and coherence of the discovered themes. Too few topics may merge distinct concepts, while too many may fragment the corpus into noisy or redundant subtopics.

Several metrics have been designed to evaluate the semantic quality of topics. The [most commonly used](https://radimrehurek.com/gensim/models/coherencemodel.html) are :
* u_mass, which measures word co-occurrence based on document frequencies within the corpus; 
* c_v, which combines sliding-window co-occurrence and normalised pointwise mutual information (NPMI) with a cosine similarity measure; 
* c_uci, which relies on pointwise mutual information (PMI) computed over word co-occurrences; 
* c_npmi, which is a normalised version of PMI that scales scores between −1 and 1 to improve interpretability.

The following cells visualise how the coherence of topics evolves as the number of topics (k) increases. For each coherence metric (u_mass, c_v, c_uci and c_npmi), we calculate the coherence score for various values of k. Since these metrics have different scales and are not directly comparable, we apply min–max normalisation independently to each metric. This enables us to compare their relative trends across different numbers of topics. The resulting plot helps to identify values of k that provide a good balance between topic granularity and semantic coherence.'



In [ ]:
from gensim.corpora import Dictionary

analyzer = tfidf_vectorizer.build_analyzer()

tokenized_texts = [
    analyzer(doc)
    for doc in df.lemmatized_text
]


dictionary = Dictionary(tokenized_texts)

# Optional but recommended:
dictionary.filter_extremes(no_below=5, no_above=0.95)


In [ ]:
from gensim.models.coherencemodel import CoherenceModel

def get_topics(model, feature_names, n_top_words):
    topics = []
    for topic_idx, topic in enumerate(model.components_):
        # Get indices of top words (descending order)
        top_indices = np.argsort(-topic)[:n_top_words]
        # Map indices to words
        top_words = [feature_names[i] for i in top_indices]
        topics.append(top_words)
    return topics

feature_names = tfidf_vectorizer.get_feature_names_out()

# for each number of topics, we will compute the 4 metrics
topic_range = [5, 10, 15, 20, 30]
coherence_metrics = ['u_mass', 'c_v', 'c_uci', 'c_npmi']
for metrics in coherence_metrics:
    results[metrics] = {}


for k in topic_range:
    # fit the model with k topics
    nmf = NMF(
        n_components=k,
        random_state=42
    )

    W = nmf.fit_transform(tfidf)
    topics = get_topics(nmf, feature_names, 10)
    for met in coherence_metrics:

        coherence_model = CoherenceModel(
            topics=topics,
            texts=tokenized_texts,
            dictionary=dictionary,
            coherence=met
        )

        coherence_score = coherence_model.get_coherence()
        results[met][k] =  coherence_score
        print(f"NMF coherence for {k} topic and metrics {met}: {coherence_score}")

In [ ]:
import numpy as np

plt.figure(figsize=(10, 6))

for met in coherence_metrics:
    ks = sorted(results[met].keys())
    scores = np.array([results[met][k] for k in ks])

    # Min-max normalize per metric
    norm_scores = (scores - scores.min()) / (scores.max() - scores.min())

    plt.plot(ks, norm_scores, marker='o', label=met)

plt.xlabel("Number of Topics (k)")
plt.ylabel("Normalized Coherence Score")
plt.title("Normalized Coherence Comparison")
plt.legend()
plt.grid(True)
plt.show()